## Cell 0 — Environment Setup
Must be the very first cell. Sets env vars before any imports.

In [1]:
import os
import sys

# Must be set before any unsloth/torch imports
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
os.environ["UNSLOTH_DISABLE_FAST_DOWNLOAD"] = "1"

import torch
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.recompile_limit = 256

print(f"Interpreter : {sys.executable}")
print(f"Torch       : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Cannot proceed.")


Interpreter : e:\Anti_gravity\Python_Installation\python.exe
Torch       : 2.12.0.dev20260408+cu128
CUDA        : True
Device      : NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM        : 12.8 GB


## Cell 1 — Clear Stale Compiled Cache
Always run this after restarting to avoid fullgraph recompile errors.

In [2]:
import shutil
import glob

cache_dir = os.path.join(os.getcwd(), "unsloth_compiled_cache")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Cleared: {cache_dir}")
else:
    print("No cache to clear")


No cache to clear


## Cell 2 — Load Base Model + LoRA Adapters
Loads base model from HuggingFace (cached after first run), then attaches your saved LoRA weights.
Gradient checkpointing disabled to prevent the fullgraph recompile crash.

In [3]:
from unsloth import FastVisionModel
from peft import set_peft_model_state_dict
import safetensors.torch

# Load base model — cached locally after first download
model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/gemma-4-e2b-it",
    load_in_4bit=True,
    max_seq_length=2048,
    use_gradient_checkpointing=False,  # Critical: disables the fullgraph recompile crash
)

# Reconstruct LoRA architecture with same config as original training
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_vision_projector=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_rslora=False,
)

# Load your saved adapter weights
adapter_path = "./pharos_gemma4_final/model/adapter_model.safetensors"
if os.path.exists(adapter_path):
    weights = safetensors.torch.load_file(adapter_path)
    set_peft_model_state_dict(model, weights)
    print("Loaded saved LoRA adapter weights from checkpoint")
else:
    print("No checkpoint found — starting from fresh LoRA init")

print(f"Model on device: {next(model.parameters()).device}")
print(f"Tokenizer type : {type(tokenizer)}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


e:\Anti_gravity\Python_Installation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0504 02:36:47.147000 47448 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0504 02:36:47.231000 47448 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti Laptop GPU. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260408+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 2011/2011 [00:16<00:00, 123.06it/s]
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Loaded saved LoRA adapter weights from checkpoint
Model on device: cuda:0
Tokenizer type : <class 'transformers.models.gemma4.processing_gemma4.Gemma4Processor'>


## Cell 3 — Load Dataset

In [4]:
import json
import traceback
from datasets import load_dataset, Dataset
from PIL import Image

print("Loading pill dataset from HuggingFace...")
raw_dataset = load_dataset("Francesco/pills-sxdht", split="train")
print(f"Loaded {len(raw_dataset)} samples")
print(f"Columns: {raw_dataset.column_names}")


Loading pill dataset from HuggingFace...
Loaded 316 samples
Columns: ['image_id', 'image', 'width', 'height', 'objects']


## Cell 4 — Labelling Prompt and Function
Fixed: inputs properly moved to cuda device. Debug mode available.

In [5]:
LABELLING_PROMPT = """Analyse this pharmaceutical pill image carefully.
Return ONLY a valid JSON object with exactly these fields, no other text:
{
  "medicine_name": "inferred name or Unknown",
  "batch_code": "visible code or UNKNOWN",
  "expiry_date": "visible date or UNKNOWN",
  "risk_score": <integer 0-100>,
  "risk_level": "<LOW|MODERATE|HIGH|CRITICAL>",
  "flags": ["specific issues observed, empty list if none"],
  "image_quality_ok": <true|false>
}

Risk level decision rules (apply strictly):
- LOW (0-30): All pills present, clearly visible, no damage, sharp image
- MODERATE (31-60): Minor blur OR 1-2 pills partially obscured OR slight discoloration
- HIGH (61-80): Multiple missing pills OR significant blur OR visible cracks/damage
- CRITICAL (81-100): Contamination visible OR completely unidentifiable OR severe damage

Default to LOW if the pills are clearly visible and intact.
Only use MODERATE if there is a specific quality issue."""


def label_sample(sample, model, tokenizer, debug=False):
    """
    Run Gemma 4 zero-shot on a single image to produce a Pharos-format JSON label.
    Falls back to a default MODERATE label if parsing fails.
    """
    try:
        image = sample["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image).convert("RGB")
        else:
            image = image.convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": LABELLING_PROMPT},
                ],
            }
        ]

        # Fixed: move each tensor individually to cuda
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
        )
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                pad_token_id=tokenizer.eos_token_id,
                temperature=0.3,
                do_sample=True,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        if debug:
            print(f"Raw response: {response}")

        json_start = response.find("{")
        json_end = response.rfind("}") + 1

        if json_start == -1 or json_end == 0:
            raise ValueError(f"No JSON found in response: {response[:200]}")

        label_json = json.loads(response[json_start:json_end])

        assert label_json.get("risk_level") in ("LOW", "MODERATE", "HIGH", "CRITICAL"), \
            f"Invalid risk_level: {label_json.get('risk_level')}"
        assert isinstance(label_json.get("risk_score"), int), \
            f"Invalid risk_score type: {type(label_json.get('risk_score'))}"

        if "flags" not in label_json or not isinstance(label_json["flags"], list):
            label_json["flags"] = []

        return json.dumps(label_json)

    except Exception as e:
        if debug:
            print(f"label_sample error: {e}")
            traceback.print_exc()
        return json.dumps({
            "medicine_name": "Unknown",
            "batch_code": "UNKNOWN",
            "expiry_date": "UNKNOWN",
            "risk_score": 50,
            "risk_level": "MODERATE",
            "flags": ["auto_label_failed"],
            "image_quality_ok": False,
        })


# Sanity check on one sample before running the full loop
print("Testing label_sample on sample 0...")
FastVisionModel.for_inference(model)
test_result = label_sample(raw_dataset[0], model, tokenizer, debug=True)
print(f"Result: {test_result}")


Testing label_sample on sample 0...


e:\Anti_gravity\Python_Installation\lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Raw response: ```json
{
  "medicine_name": "Unknown",
  "batch_code": "UNKNOWN",
  "expiry_date": "UNKNOWN",
  "risk_score": 20,
  "risk_level": "LOW",
  "flags": [
    "The image displays multiple pharmaceutical pills, likely tablets, stacked together. They appear intact and the image quality is good, allowing for general observation but not specific identification of the drug."
  ],
  "image_quality_ok": true
}
```
Result: {"medicine_name": "Unknown", "batch_code": "UNKNOWN", "expiry_date": "UNKNOWN", "risk_score": 20, "risk_level": "LOW", "flags": ["The image displays multiple pharmaceutical pills, likely tablets, stacked together. They appear intact and the image quality is good, allowing for general observation but not specific identification of the drug."], "image_quality_ok": true}


## Cell 5 — Label All Samples

In [6]:
FastVisionModel.for_inference(model)

SAMPLE_SIZE = len(raw_dataset)
subset = raw_dataset.select(range(SAMPLE_SIZE))

labelled_data = []
failed_count = 0

for i, sample in enumerate(subset):
    label = label_sample(sample, model, tokenizer)
    parsed = json.loads(label)

    if "auto_label_failed" in pased["flags"]:
        failed_count += 1

    image = sample["image"]r
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert("RGB")

    labelled_data.append({"image": image, "label": label})

    if (i + 1) % 20 == 0:
        risk = parsed["risk_level"]
        print(f"Labelled {i + 1}/{SAMPLE_SIZE} — last risk_level: {risk} | failed so far: {failed_count}")

print(f"Done: {len(labelled_data)} samples labelled, {failed_count} failed")


Labelled 20/316 — last risk_level: LOW | failed so far: 0
Labelled 40/316 — last risk_level: LOW | failed so far: 0
Labelled 60/316 — last risk_level: LOW | failed so far: 0
Labelled 80/316 — last risk_level: LOW | failed so far: 0
Labelled 100/316 — last risk_level: LOW | failed so far: 0
Labelled 120/316 — last risk_level: LOW | failed so far: 0
Labelled 140/316 — last risk_level: LOW | failed so far: 0
Labelled 160/316 — last risk_level: LOW | failed so far: 0
Labelled 180/316 — last risk_level: LOW | failed so far: 0
Labelled 200/316 — last risk_level: LOW | failed so far: 0
Labelled 220/316 — last risk_level: LOW | failed so far: 0
Labelled 240/316 — last risk_level: LOW | failed so far: 0
Labelled 260/316 — last risk_level: LOW | failed so far: 0
Labelled 280/316 — last risk_level: LOW | failed so far: 0
Labelled 300/316 — last risk_level: LOW | failed so far: 0
Done: 316 samples labelled, 0 failed


## Cell 6 — Filter and Rebalance Dataset
Fixed: ZeroDivisionError when all samples fail.

In [7]:
import random
from collections import Counter

# Filter failed labels
clean_data = [
    d for d in labelled_data
    if "auto_label_failed" not in json.loads(d["label"])["flags"]
]
failed = len(labelled_data) - len(clean_data)
print(f"Filtered: {failed} failed, {len(clean_data)} clean samples remain")

if len(clean_data) == 0:
    raise RuntimeError(
        "All samples failed labelling. Re-run Cell 4 with debug=True on a single sample "
        "to find the actual error before proceeding."
    )

risk_counts = Counter(json.loads(d["label"])["risk_level"] for d in clean_data)
print("\nLabel distribution before rebalancing:")
for level in ["LOW", "MODERATE", "HIGH", "CRITICAL"]:
    count = risk_counts.get(level, 0)
    pct = count / len(clean_data) * 100
    bar = "#" * int(pct / 2)
    print(f"  {level:10s}: {count:4d} ({pct:5.1f}%) {bar}")

dominant_class_pct = max(risk_counts.values()) / len(clean_data)
if dominant_class_pct > 0.60:
    print("\nClass imbalance detected — applying capped oversampling")
    by_class = {level: [] for level in ["LOW", "MODERATE", "HIGH", "CRITICAL"]}
    for d in clean_data:
        level = json.loads(d["label"])["risk_level"]
        by_class[level].append(d)

    max_class = max(len(v) for v in by_class.values())
    min_per_class = max(10, max_class // 3)

    balanced = []
    for level, samples in by_class.items():
        if len(samples) == 0:
            continue
        if len(samples) < min_per_class:
            oversampled = samples + random.choices(samples, k=min_per_class - len(samples))
            balanced.extend(oversampled)
            print(f"  {level}: {len(samples)} -> {len(oversampled)} (oversampled)")
        else:
            balanced.extend(samples)
            print(f"  {level}: {len(samples)} (unchanged)")

    random.shuffle(balanced)
    clean_data = balanced
    print(f"\nRebalanced dataset: {len(clean_data)} samples")
else:
    print("\nDistribution acceptable — no rebalancing needed")

final_counts = Counter(json.loads(d["label"])["risk_level"] for d in clean_data)
print("\nFinal label distribution:")
for level in ["LOW", "MODERATE", "HIGH", "CRITICAL"]:
    count = final_counts.get(level, 0)
    pct = count / len(clean_data) * 100
    print(f"  {level:10s}: {count:4d} ({pct:5.1f}%)")


Filtered: 0 failed, 316 clean samples remain

Label distribution before rebalancing:
  LOW       :  316 (100.0%) ##################################################
  MODERATE  :    0 (  0.0%) 
  HIGH      :    0 (  0.0%) 
  CRITICAL  :    0 (  0.0%) 

Class imbalance detected — applying capped oversampling
  LOW: 316 (unchanged)

Rebalanced dataset: 316 samples

Final label distribution:
  LOW       :  316 (100.0%)
  MODERATE  :    0 (  0.0%)
  HIGH      :    0 (  0.0%)
  CRITICAL  :    0 (  0.0%)


## Cell 7 — Build and Format Dataset

In [8]:
pharos_dataset = Dataset.from_dict({
    "image": [d["image"] for d in clean_data],
    "label": [d["label"] for d in clean_data],
})

split = pharos_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train : {len(train_dataset)} samples")
print(f"Eval  : {len(eval_dataset)} samples")

INFERENCE_PROMPT = (
    "Analyse this pharmaceutical blister pack image. "
    "Return a JSON object with: medicine_name, batch_code, "
    "expiry_date, risk_score (0-100), risk_level "
    "(LOW/MODERATE/HIGH/CRITICAL), flags (list), "
    "and image_quality_ok (bool)."
)


def format_conversation(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": sample["image"]},
                    {"type": "text", "text": INFERENCE_PROMPT},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": sample["label"]}],
            },
        ]
    }


train_formatted = train_dataset.map(format_conversation)
eval_formatted  = eval_dataset.map(format_conversation)

print("Datasets formatted for training")
print(f"Columns: {train_formatted.column_names}")
print(f"\nSample label: {train_dataset[0]['label']}")


Train : 284 samples
Eval  : 32 samples


Map: 100%|██████████| 32/32 [00:00<00:00, 89.00 examples/s]

Datasets formatted for training
Columns: ['image', 'label', 'messages']

Sample label: {"medicine_name": "Unknown", "batch_code": "UNKNOWN", "expiry_date": "UNKNOWN", "risk_score": 20, "risk_level": "LOW", "flags": ["The image shows multiple pharmaceutical pills, likely tablets, stacked together. They appear intact and the image quality is sufficient for general observation, although specific identification is impossible without markings or context."], "image_quality_ok": true}


## Cell 8 — Fine-tune for 7 Hours
Key fixes: gradient_checkpointing=False stops the fullgraph recompile crash.
More epochs and frequent checkpoints optimised for a long unattended run.

In [9]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator
from transformers import EarlyStoppingCallback

# Patch compiled cache if it exists to remove fullgraph=True
import re
import glob

cache_dir = os.path.join(os.getcwd(), "unsloth_compiled_cache")
for fpath in glob.glob(os.path.join(cache_dir, "*.py")):
    with open(fpath, "r", encoding="utf-8") as f:
        content = f.read()
    if "fullgraph=True" in content:
        patched = content.replace("fullgraph=True", "fullgraph=False")
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(patched)
        print(f"Patched fullgraph=True in {os.path.basename(fpath)}")

# Enable training mode — gradient checkpointing disabled to prevent recompile crash
FastVisionModel.for_training(model, use_gradient_checkpointing=False)

training_args = SFTConfig(
    output_dir="./pharos_gemma4_finetuned",
    num_train_epochs=8,                       # ~7 hours at ~5 min/epoch on RTX 5070 Ti
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,             # Critical: prevents fullgraph recompile crash
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=5,
    save_steps=50,
    save_total_limit=5,                       # Keep last 5 checkpoints to save disk space
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    seed=42,
    max_seq_length=2048,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),  # Stop if no improvement for 5 evals
    ],
)

# Log GPU state before starting
gpu_stats = torch.cuda.get_device_properties(0)
reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total    = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU    : {gpu_stats.name}")
print(f"VRAM   : {reserved} GB reserved of {total} GB total")
print(f"Steps  : {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto'}")
print("\nTrainer ready — starting training")

trainer.train()


Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


GPU    : NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM   : 7.76 GB reserved of 11.94 GB total
Steps  : auto

Trainer ready — starting training


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 284 | Num Epochs = 8 | Total steps = 288
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,859,840 of 5,153,037,856 (0.58% trained)


Step,Training Loss,Validation Loss
50,0.095578,0.084941
100,0.044547,0.059850
150,0.043562,0.055522
200,0.042745,0.052817
250,0.040941,0.051032
288,0.036699,0.051042


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in ./pharos_gemma4_finetuned\checkpoint-50\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./pharos_gemma4_finetuned\checkpoint-100\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./pharos_gemma4_finetuned\checkpoint-150\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./pharos_gemma4_finetuned\checkpoint-200\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./pharos_gemma4_finetuned\checkpoint-250\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./pharos_gemma4_finetuned\checkpoint-288\tokenizer_config.json.


TrainOutput(global_step=288, training_loss=0.1783371919559108, metrics={'train_runtime': 59953.0418, 'train_samples_per_second': 0.038, 'train_steps_per_second': 0.005, 'total_flos': 1.3807074086668032e+16, 'train_loss': 0.1783371919559108, 'epoch': 8.0})

## Cell 9 — Save Final Model

In [10]:
from pathlib import Path

def save_finetuned_model(model, tokenizer, output_path="./pharos_gemma4_final_v2"):
    Path(output_path).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(f"{output_path}/model")
    tokenizer.save_pretrained(f"{output_path}/tokenizer")

    config = {
        "base_model":                "unsloth/gemma-4-e2b-it",
        "finetune_vision_layers":    True,
        "finetune_vision_projector": True,
        "max_seq_length":            2048,
        "lora_r":                    16,
        "lora_alpha":                32,
        "lora_dropout":              0.05,
    }

    with open(f"{output_path}/adapter_config.json", "w") as f:
        json.dump(config, f, indent=2)

    print(f"Model saved to {output_path}/")
    print(f"  Weights   : {output_path}/model")
    print(f"  Tokenizer : {output_path}/tokenizer")
    print(f"  Config    : {output_path}/adapter_config.json")


save_finetuned_model(model, tokenizer)


Unsloth: Restored added_tokens_decoder metadata in ./pharos_gemma4_final_v2/tokenizer\tokenizer_config.json.


Model saved to ./pharos_gemma4_final_v2/
  Weights   : ./pharos_gemma4_final_v2/model
  Tokenizer : ./pharos_gemma4_final_v2/tokenizer
  Config    : ./pharos_gemma4_final_v2/adapter_config.json


## Cell 10 — Evaluate Fine-tuned Model

In [11]:
from collections import defaultdict

def evaluate_model(model, tokenizer, eval_dataset, num_samples=None):
    FastVisionModel.for_inference(model)

    if num_samples is None:
        num_samples = len(eval_dataset)

    correct = 0
    total = 0
    per_class = defaultdict(lambda: {"correct": 0, "total": 0})
    parse_failures = 0

    with torch.no_grad():
        for sample in eval_dataset.select(range(min(num_samples, len(eval_dataset)))):
            image = sample["image"]
            if not isinstance(image, Image.Image):
                image = Image.fromarray(image).convert("RGB")

            label = json.loads(sample["label"])
            true_level = label.get("risk_level", "UNKNOWN")

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": INFERENCE_PROMPT},
                    ],
                }
            ]

            # Fixed: move inputs to cuda individually
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_tensors="pt",
                return_dict=True,
            )
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                pad_token_id=tokenizer.eos_token_id,
            )

            new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
            response = tokenizer.decode(new_tokens, skip_special_tokens=True)

            try:
                json_start = response.find("{")
                json_end = response.rfind("}") + 1
                pred = json.loads(response[json_start:json_end])
                pred_level = pred.get("risk_level", "UNKNOWN")

                per_class[true_level]["total"] += 1
                if pred_level == true_level:
                    correct += 1
                    per_class[true_level]["correct"] += 1
            except Exception:
                parse_failures += 1
                per_class[true_level]["total"] += 1

            total += 1

    accuracy = correct / total if total > 0 else 0.0
    print(f"Overall risk_level accuracy : {accuracy:.2%} ({correct}/{total})")
    print(f"JSON parse failures         : {parse_failures}/{total}")
    print("\nPer-class accuracy:")
    for level in ["LOW", "MODERATE", "HIGH", "CRITICAL"]:
        stats = per_class.get(level, {"correct": 0, "total": 0})
        if stats["total"] > 0:
            cls_acc = stats["correct"] / stats["total"]
            print(f"  {level:10s}: {cls_acc:.2%} ({stats['correct']}/{stats['total']})")
        else:
            print(f"  {level:10s}: no samples in eval set")

    return accuracy


evaluate_model(model, tokenizer, eval_dataset)


e:\Anti_gravity\Python_Installation\lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Overall risk_level accuracy : 100.00% (32/32)
JSON parse failures         : 0/32

Per-class accuracy:
  LOW       : 100.00% (32/32)
  MODERATE  : no samples in eval set
  HIGH      : no samples in eval set
  CRITICAL  : no samples in eval set


1.0